# Handcrafted two-arm URDF render debug

基于已有 [`result.json`](ca-pi-xie-device-tools/reconstruct_two_arm.ipynb:56) / [`rise2_calib_20260104.npy`](build_rise2_calib_from_json.py:1) / converted [`lowdim/*.npy`](dataset/data_utils.py:18) 手搓一套双臂渲染与内联可视化。

In [ ]:
import os
import sys
import json
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation as R
from omegaconf import OmegaConf

pkg_root = os.path.abspath('airexo')
if pkg_root not in sys.path:
    sys.path.insert(0, pkg_root)
for _m in [m for m in list(sys.modules.keys()) if m == 'airexo' or m.startswith('airexo.')]:
    del sys.modules[_m]

from mask.renderer import ArmOnlyRobotRenderer

TASK_ROOT = Path('/data/haoxiang/data/task0012_260321/task0012_toys_basket_converted')
SCENE_NAME = 'scene_0001'
FRAME_STEM = '1736320913189'  # 可改成 lowdim/ 下实际存在的任意一帧
CALIB_TIMESTAMP = '20260104'
GLOBAL_SERIAL = '104122060902'
LEFT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/left_global_20260104/result.json')
RIGHT_JSON = Path('/data/haoxiang/data/task0012_260321/calib/right_global_20260104/result.json')

scene_path = TASK_ROOT / 'train' / SCENE_NAME
cam_path = scene_path / f'cam_{GLOBAL_SERIAL}'
lowdim_path = scene_path / 'lowdim' / f'{FRAME_STEM}.npy'
rgb_path = cam_path / 'color' / f'{FRAME_STEM}.png'
depth_path = cam_path / 'depth' / f'{FRAME_STEM}.png'
calib_npy_path = TASK_ROOT / 'calib' / f'rise2_calib_{CALIB_TIMESTAMP}.npy'

assert lowdim_path.exists(), lowdim_path
assert rgb_path.exists(), rgb_path
assert depth_path.exists(), depth_path
assert calib_npy_path.exists(), calib_npy_path
print('scene_path =', scene_path)
print('frame =', FRAME_STEM)
print('rgb_path =', rgb_path)
print('depth_path =', depth_path)


In [ ]:
def pose7_to_mat_wxyz(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64)
    t = pose7[:3]
    qw, qx, qy, qz = pose7[3:]
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    mat[:3, 3] = t
    return mat

def pose7_to_mat_xyzw(pose7):
    pose7 = np.asarray(pose7, dtype=np.float64)
    t = pose7[:3]
    qx, qy, qz, qw = pose7[3:]
    mat = np.eye(4, dtype=np.float64)
    mat[:3, :3] = R.from_quat([qx, qy, qz, qw]).as_matrix()
    mat[:3, 3] = t
    return mat

def invert_T(T):
    T = np.asarray(T, dtype=np.float64)
    out = np.eye(4, dtype=np.float64)
    out[:3,:3] = T[:3,:3].T
    out[:3,3] = -T[:3,:3].T @ T[:3,3]
    return out

def load_pose_in_link(json_path):
    data = json.loads(Path(json_path).read_text())
    pose = np.asarray(data['pose_in_link'], dtype=np.float64)
    return pose7_to_mat_wxyz(pose), data

left_cam_to_left_base, left_json = load_pose_in_link(LEFT_JSON)
right_cam_to_right_base, right_json = load_pose_in_link(RIGHT_JSON)

rise2_calib = np.load(calib_npy_path, allow_pickle=True).item()
intrinsic = np.asarray(rise2_calib['intrinsics'][GLOBAL_SERIAL], dtype=np.float64)
rise2_left = np.asarray(rise2_calib['camera_to_robot_left'][GLOBAL_SERIAL], dtype=np.float64)
rise2_right = np.asarray(rise2_calib['camera_to_robot_right'][GLOBAL_SERIAL], dtype=np.float64)

print('left_json camera_to_left_base =\n', left_cam_to_left_base)
print('right_json camera_to_right_base =\n', right_cam_to_right_base)
print('rise2 camera_to_robot_left =\n', rise2_left)
print('rise2 camera_to_robot_right =\n', rise2_right)
print('left diff max =', np.abs(left_cam_to_left_base - rise2_left).max())
print('right diff max =', np.abs(right_cam_to_right_base - rise2_right).max())


In [ ]:
lowdim = np.load(lowdim_path, allow_pickle=True).item()
for k, v in lowdim.items():
    print(k, np.asarray(v).shape)

# 这里没有原始关节角，只能手搓：
# 用 0 关节角作为初始姿态，只利用 JSON / RISE2 标定把双臂放进相机坐标系，
# 先验证外参链和渲染方向是否一致。
left_joint = np.zeros(8, dtype=np.float32)
right_joint = np.zeros(8, dtype=np.float32)
left_joint[-1] = float(np.asarray(lowdim['gripper_left'])[0]) if 'gripper_left' in lowdim else 0.05
right_joint[-1] = float(np.asarray(lowdim['gripper_right'])[0]) if 'gripper_right' in lowdim else 0.05

print('left_joint =', left_joint)
print('right_joint =', right_joint)


In [ ]:
left_cfg = OmegaConf.load('airexo/airexo/configs/joint/left/robot.yaml')
right_cfg = OmegaConf.load('airexo/airexo/configs/joint/right/robot.yaml')

rgb_bgr = cv2.imread(str(rgb_path), cv2.IMREAD_COLOR)
rgb = cv2.cvtColor(rgb_bgr, cv2.COLOR_BGR2RGB)
depth_raw = cv2.imread(str(depth_path), cv2.IMREAD_UNCHANGED)

height, width = rgb.shape[:2]

# 直接参考 reconstruct_two_arm.ipynb 的 renderer 调用风格，
# 这里只是把 cam_to_base 手工指定成左臂 / 右臂各自的一套，分别渲染再叠加。
renderer_left = ArmOnlyRobotRenderer(
    left_joint_cfgs=left_cfg,
    right_joint_cfgs=right_cfg,
    cam_to_base=left_cam_to_left_base,
    intrinsic=intrinsic,
    urdf_file='airexo/airexo/urdf_models/robot/robot_inhand.urdf',
    width=width,
    height=height,
    near_plane=0.01,
    far_plane=100.0,
)
renderer_left.update_joints(left_joint, right_joint)
rendered_left_rgb = renderer_left.render_image()
rendered_left_depth = renderer_left.render_depth()
rendered_left_mask = renderer_left.render_mask(depth=rendered_left_depth)

renderer_right = ArmOnlyRobotRenderer(
    left_joint_cfgs=left_cfg,
    right_joint_cfgs=right_cfg,
    cam_to_base=right_cam_to_right_base,
    intrinsic=intrinsic,
    urdf_file='airexo/airexo/urdf_models/robot/robot_inhand.urdf',
    width=width,
    height=height,
    near_plane=0.01,
    far_plane=100.0,
)
renderer_right.update_joints(left_joint, right_joint)
rendered_right_rgb = renderer_right.render_image()
rendered_right_depth = renderer_right.render_depth()
rendered_right_mask = renderer_right.render_mask(depth=rendered_right_depth)

overlay = rgb.copy()
mask_left = rendered_left_mask > 0
mask_right = rendered_right_mask > 0
overlay[mask_left] = (0.6 * rendered_left_rgb[mask_left] + 0.4 * overlay[mask_left]).astype(np.uint8)
overlay[mask_right] = (0.6 * rendered_right_rgb[mask_right] + 0.4 * overlay[mask_right]).astype(np.uint8)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0, 0].imshow(rgb); axes[0, 0].set_title('real rgb'); axes[0, 0].axis('off')
axes[0, 1].imshow(rendered_left_mask, cmap='gray'); axes[0, 1].set_title('left render mask'); axes[0, 1].axis('off')
axes[0, 2].imshow(rendered_right_mask, cmap='gray'); axes[0, 2].set_title('right render mask'); axes[0, 2].axis('off')
axes[1, 0].imshow(overlay); axes[1, 0].set_title('overlay handcrafted'); axes[1, 0].axis('off')
axes[1, 1].imshow(depth_raw, cmap='plasma'); axes[1, 1].set_title('real depth raw'); axes[1, 1].axis('off')
axes[1, 2].imshow((mask_left.astype(np.uint8) * 127 + mask_right.astype(np.uint8) * 128), cmap='gray'); axes[1, 2].set_title('combined mask'); axes[1, 2].axis('off')
plt.tight_layout()
plt.show()


## 说明

这份 notebook 不是严格恢复 AirExo 原始 calibration 资产，而是：
- 直接使用左右 [`result.json`](ca-pi-xie-device-tools/reconstruct_two_arm.ipynb:56) 的 [`pose_in_link`](ca-pi-xie-device-tools/reconstruct_two_arm.ipynb:57)
- 与 [`rise2_calib_20260104.npy`](build_rise2_calib_from_json.py:1) 中的 [`camera_to_robot_left/right`](dataset/projector.py:92) 互相校验
- 再手工喂给 renderer 做内联叠加调试

如果后续要继续逼近真实姿态，需要再补：
1. 真实双臂关节角来源
2. 左右基座之间的统一世界系关系
3. renderer 对单/双臂 `cam_to_base` 语义的进一步核对
